# STAC Metadata

OceanStream generates [SpatioTemporal Asset Catalog (STAC)](https://stacspec.org/) 1.0 metadata for all processed data.

## Output Structure
```
output_dir/campaign_id/
├── lat_bin=X/lon_bin=Y/*.parquet   # Hive-partitioned GeoParquet
├── tiles/track.pmtiles             # Vector tiles (optional)
└── stac/
    ├── collection.json             # STAC Collection
    └── items/
        └── *.json                  # STAC Items (per-file)
```

In [ ]:
# Setup
import sys
import json
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

## Generate Sample Data

In [ ]:
from oceanstream import convert
import tempfile

input_dir = project_root / "oceanstream" / "tests" / "data" / "raw_data"
output_dir = Path(tempfile.mkdtemp()) / "stac_demo"

convert(
    provider="saildrone",
    input_source=input_dir,
    output_dir=output_dir,
    campaign_id="stac_demo",
    generate_pmtiles=False,
    verbose=True,
    yes=True,
)

## Exploring the STAC Collection

In [ ]:
stac_dir = output_dir / "stac_demo" / "stac"
collection_path = stac_dir / "collection.json"

with open(collection_path) as f:
    collection = json.load(f)

print("📦 STAC Collection")
print(f"   ID: {collection['id']}")
print(f"   Title: {collection.get('title', 'N/A')}")
print(f"   Description: {collection.get('description', 'N/A')[:80]}...")

In [ ]:
# Spatial extent
bbox = collection['extent']['spatial']['bbox'][0]
print(f"📍 Spatial Extent:")
print(f"   West: {bbox[0]:.4f}, South: {bbox[1]:.4f}")
print(f"   East: {bbox[2]:.4f}, North: {bbox[3]:.4f}")

In [ ]:
# Temporal extent
temporal = collection['extent']['temporal']['interval'][0]
print(f"⏰ Temporal Extent:")
print(f"   Start: {temporal[0]}")
print(f"   End: {temporal[1]}")

In [ ]:
# Platforms (multi-platform support)
platforms = collection.get('summaries', {}).get('platforms', [])
print(f"🚢 Platforms ({len(platforms)}):")
for p in platforms:
    print(f"   • {p.get('id', 'unknown')}: {p.get('type', 'N/A')} ({p.get('row_count', 0):,} rows)")

In [ ]:
# Instruments/Sensors
instruments = collection.get('summaries', {}).get('instruments', [])
print(f"🔬 Instruments ({len(instruments)}):")
for i in instruments[:5]:  # Show first 5
    print(f"   • {i}")

## STAC Items

In [ ]:
items_dir = stac_dir / "items"
item_files = list(items_dir.glob("*.json")) if items_dir.exists() else []

print(f"📄 STAC Items: {len(item_files)}")
if item_files:
    with open(item_files[0]) as f:
        item = json.load(f)
    print(f"\nFirst item: {item['id']}")
    print(f"   Geometry type: {item['geometry']['type']}")
    print(f"   Properties: {list(item['properties'].keys())}")

## Using STAC with pystac

```python
import pystac

# Load collection
collection = pystac.Collection.from_file("stac/collection.json")

# Get all items
for item in collection.get_all_items():
    print(f"{item.id}: {item.datetime}")
    
# Access assets
for key, asset in collection.assets.items():
    print(f"{key}: {asset.href}")
```